# 売買方向の非対称性・別閾値

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 20


In [ ]:
# ============================================================
# BUY / SELL 非対称性 診断実験
#
# 目的
# 1. SELLの高PFが複数Foldで再現するか
# 2. BUY偏重がモデル由来か、相場自体の偏りか
# 3. BUY/SELLがどのRegimeで強いか
# 4. Bootstrapで平均リターンの不確実性を確認
#
# 前提:
# 前のセルで以下が定義済み
# load_bars
# prepare_data
# outer_folds
# fit_base_models
# predict_base_models
# make_signals
# run_backtest
# strategy_stats
# MOVE_PROB_LIST
# DIRECTION_PROB_LIST
# TP_LIST
# SL_LIST
# MIN_VALIDATION_TRADES
# HORIZON_BARS
# ============================================================


from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 最新の固定CSVを自動で探す
# ============================================================

def find_latest_snapshot():
    """
    fx_experiment_runs の中から
    最新の usdjpy_5m.csv を探す。
    """

    root = Path.cwd() / "fx_experiment_runs"

    files = list(
        root.glob(
            "*/usdjpy_5m.csv"
        )
    )

    if not files:
        raise FileNotFoundError(
            "fx_experiment_runs 内に usdjpy_5m.csv がありません。"
        )

    latest = max(
        files,
        key=lambda p: p.stat().st_mtime
    )

    return latest


csv_path = find_latest_snapshot()

print(
    "使用する固定CSV:"
)

print(
    csv_path
)


# ============================================================
# 2. データ読み込み
# ============================================================

bars = load_bars(
    csv_path
)

data = prepare_data(
    bars
)

print()
print(
    "5分足:",
    len(bars)
)

print(
    "使用可能データ:",
    len(data)
)


# ============================================================
# 3. Regime特徴を追加
#
# ここでは未来は使わず、
# シグナル時点までの特徴量だけで分類する
# ============================================================

def add_regime_columns(frame):

    df = frame.copy()

    # ----------------------------
    # Volatility
    # ----------------------------

    vol_median = (
        df["volatility_1h"]
        .median()
    )

    df["vol_regime"] = np.where(
        df["volatility_1h"]
        >= vol_median,
        "HIGH_VOL",
        "LOW_VOL"
    )

    # ----------------------------
    # Trend / Range
    # ADX 25以上をTrendと仮定
    # ----------------------------

    df["trend_regime"] = np.where(
        df["ADX14"]
        >= 25,
        "TREND",
        "RANGE"
    )

    # ----------------------------
    # 上昇 / 下降トレンド
    # MA50 slopeで簡単に分類
    # ----------------------------

    df["trend_direction"] = np.where(
        df["MA50_slope"] > 0,
        "UP_TREND",
        "DOWN_TREND"
    )

    # ----------------------------
    # Session
    #
    # Asia/Tokyo時刻を前提
    #
    # 東京  9-15
    # London 16-23
    # NY その他
    #
    # 厳密なDST対応ではないため
    # 今回は診断用の大分類
    # ----------------------------

    hour = df.index.hour

    df["session"] = np.select(
        [
            (hour >= 9) & (hour < 16),
            (hour >= 16) & (hour < 24),
        ],
        [
            "TOKYO",
            "LONDON",
        ],
        default="NY_OTHER"
    )

    return df


data = add_regime_columns(
    data
)


# ============================================================
# 4. ValidationでBASE設定だけを選択
#
# Test結果は一切使わない
# ============================================================

def choose_base_setting(
    bars,
    fold,
    probabilities
):

    val_end = (
        fold.validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    best = None
    best_score = -np.inf

    for (
        move_t,
        direction_t,
        tp,
        sl
    ) in itertools.product(
        MOVE_PROB_LIST,
        DIRECTION_PROB_LIST,
        TP_LIST,
        SL_LIST
    ):

        signals = make_signals(
            *probabilities,
            move_t,
            direction_t
        )

        trades = run_backtest(
            bars,
            fold.validation,
            signals,
            tp,
            sl,
            end_time=val_end
        )

        if (
            len(trades)
            <
            MIN_VALIDATION_TRADES
        ):
            continue

        score = (
            trades["net_return"].mean()
            *
            np.sqrt(
                len(trades)
            )
        )

        if score > best_score:

            best_score = score

            best = {
                "move_threshold":
                    move_t,

                "direction_threshold":
                    direction_t,

                "tp":
                    tp,

                "sl":
                    sl,

                "validation_score":
                    score,

                "validation_trades":
                    len(trades)
            }

    return best


# ============================================================
# 5. 全FoldでBASEだけ再評価
#
# BUY / SELL候補数
# 実行取引
# p_up / p_down
# 相場の実際の上下
# を保存する
# ============================================================

fold_rows = []

all_trades = []

prediction_rows = []


for fold in outer_folds(
    data
):

    print()
    print(
        "=================================="
    )

    print(
        f"Fold {fold.number}"
    )

    print(
        "=================================="
    )

    if (
        len(fold.train) < 500
        or
        len(fold.validation) == 0
        or
        len(fold.test) == 0
    ):

        print(
            "データ不足のためskip"
        )

        continue

    # --------------------------------
    # Validation用モデル
    # --------------------------------

    validation_model = fit_base_models(
        fold.core,
        trees=300
    )

    if validation_model is None:

        print(
            "モデル学習不可"
        )

        continue

    validation_prob = predict_base_models(
        validation_model,
        fold.validation
    )

    best = choose_base_setting(
        bars,
        fold,
        validation_prob
    )

    if best is None:

        print(
            "Validationで設定を選べませんでした"
        )

        continue

    print(
        "選択設定:",
        best
    )

    # --------------------------------
    # Test直前まで全部で再学習
    # --------------------------------

    final_model = fit_base_models(
        fold.train,
        trees=400
    )

    if final_model is None:

        continue

    (
        p_move,
        p_up,
        p_down
    ) = predict_base_models(
        final_model,
        fold.test
    )

    # --------------------------------
    # BASEシグナル
    # --------------------------------

    signals = make_signals(
        p_move,
        p_up,
        p_down,
        best["move_threshold"],
        best["direction_threshold"]
    )

    test_end = (
        fold.test.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    trades = run_backtest(
        bars,
        fold.test,
        signals,
        best["tp"],
        best["sl"],
        end_time=test_end
    )

    trades["fold"] = (
        fold.number
    )

    all_trades.append(
        trades
    )

    # ========================================================
    # 全Test時点の予測を保存
    # ========================================================

    pred = pd.DataFrame(
        {
            "time":
                fold.test.index,

            "fold":
                fold.number,

            "p_move":
                p_move,

            "p_up":
                p_up,

            "p_down":
                p_down,

            "signal":
                signals,

            "actual_future_return":
                fold.test[
                    "future_return"
                ].values,

            "actual_up":
                (
                    fold.test[
                        "future_return"
                    ].values
                    > 0
                ).astype(int),

            "volatility_1h":
                fold.test[
                    "volatility_1h"
                ].values,

            "ADX14":
                fold.test[
                    "ADX14"
                ].values,

            "MA50_slope":
                fold.test[
                    "MA50_slope"
                ].values,

            "vol_regime":
                fold.test[
                    "vol_regime"
                ].values,

            "trend_regime":
                fold.test[
                    "trend_regime"
                ].values,

            "trend_direction":
                fold.test[
                    "trend_direction"
                ].values,

            "session":
                fold.test[
                    "session"
                ].values,
        }
    )

    prediction_rows.append(
        pred
    )

    # ========================================================
    # Fold別 BUY / SELL
    # ========================================================

    row = {
        "fold":
            fold.number,

        "move_threshold":
            best[
                "move_threshold"
            ],

        "direction_threshold":
            best[
                "direction_threshold"
            ],

        "tp":
            best[
                "tp"
            ],

        "sl":
            best[
                "sl"
            ],
    }

    for side in [
        "ALL",
        "BUY",
        "SELL"
    ]:

        if side == "ALL":

            subset = (
                trades
            )

        else:

            subset = trades.loc[
                trades[
                    "direction"
                ]
                == side
            ]

        stats = strategy_stats(
            subset[
                "net_return"
            ]
        )

        for key, value in stats.items():

            row[
                f"{side.lower()}_{key}"
            ] = value

    fold_rows.append(
        row
    )

    print(
        "BUY:",
        row[
            "buy_trades"
        ],
        "平均:",
        row[
            "buy_avg_return"
        ],
        "PF:",
        row[
            "buy_profit_factor"
        ]
    )

    print(
        "SELL:",
        row[
            "sell_trades"
        ],
        "平均:",
        row[
            "sell_avg_return"
        ],
        "PF:",
        row[
            "sell_profit_factor"
        ]
    )


# ============================================================
# 6. 結合
# ============================================================

fold_results = pd.DataFrame(
    fold_rows
)

trades = pd.concat(
    all_trades,
    ignore_index=True
)

predictions = pd.concat(
    prediction_rows,
    ignore_index=True
)


# ============================================================
# 7. BUY / SELL総合成績
# ============================================================

print()
print(
    "=================================="
)

print(
    "BUY / SELL 総合"
)

print(
    "=================================="
)


overall_rows = []

for side in [
    "ALL",
    "BUY",
    "SELL"
]:

    if side == "ALL":

        subset = trades

    else:

        subset = trades.loc[
            trades[
                "direction"
            ]
            == side
        ]

    stats = strategy_stats(
        subset[
            "net_return"
        ]
    )

    overall_rows.append(
        {
            "side":
                side,

            **stats
        }
    )


overall = pd.DataFrame(
    overall_rows
)

shown = overall.copy()

for col in [
    "win_rate",
    "avg_return",
    "max_dd",
    "total_growth"
]:

    shown[
        col
    ] = (
        shown[
            col
        ]
        * 100
    )


print(
    shown.to_string(
        index=False
    )
)


# ============================================================
# 8. モデルはBUYに偏っているか？
# ============================================================

print()
print(
    "=================================="
)

print(
    "シグナル偏り診断"
)

print(
    "=================================="
)


buy_candidates = (
    predictions[
        "signal"
    ]
    == 1
).sum()

sell_candidates = (
    predictions[
        "signal"
    ]
    == -1
).sum()

wait_candidates = (
    predictions[
        "signal"
    ]
    == 0
).sum()


print(
    "BUY候補:",
    buy_candidates
)

print(
    "SELL候補:",
    sell_candidates
)

print(
    "WAIT:",
    wait_candidates
)

print()

print(
    "平均p_up:",
    predictions[
        "p_up"
    ].mean()
)

print(
    "平均p_down:",
    predictions[
        "p_down"
    ].mean()
)

print()

print(
    "実際の上昇率:",
    predictions[
        "actual_up"
    ].mean()
)


# ============================================================
# 9. Fold別
# ============================================================

print()
print(
    "=================================="
)

print(
    "Fold別 BUY / SELL"
)

print(
    "=================================="
)


fold_show = fold_results[
    [
        "fold",

        "buy_trades",
        "buy_win_rate",
        "buy_avg_return",
        "buy_profit_factor",

        "sell_trades",
        "sell_win_rate",
        "sell_avg_return",
        "sell_profit_factor"
    ]
].copy()


fold_show[
    "buy_win_rate"
] *= 100

fold_show[
    "sell_win_rate"
] *= 100

fold_show[
    "buy_avg_return"
] *= 100

fold_show[
    "sell_avg_return"
] *= 100


print(
    fold_show.to_string(
        index=False
    )
)


# ============================================================
# 10. Regime別分析
# ============================================================

# 取引にシグナル時点の環境を結合
prediction_lookup = predictions[
    [
        "fold",
        "time",
        "vol_regime",
        "trend_regime",
        "trend_direction",
        "session"
    ]
].copy()

prediction_lookup = prediction_lookup.rename(
    columns={
        "time":
            "signal_time"
    }
)

prediction_lookup[
    "signal_time"
] = pd.to_datetime(
    prediction_lookup[
        "signal_time"
    ],
    utc=True
)

trades[
    "signal_time"
] = pd.to_datetime(
    trades[
        "signal_time"
    ],
    utc=True
)

trade_regime = trades.merge(
    prediction_lookup,
    on=[
        "fold",
        "signal_time"
    ],
    how="left",
    validate="many_to_one"
)


def regime_table(
    df,
    column
):

    rows = []

    for regime in (
        df[
            column
        ]
        .dropna()
        .unique()
    ):

        for side in [
            "BUY",
            "SELL"
        ]:

            subset = df.loc[
                (
                    df[
                        column
                    ]
                    == regime
                )
                &
                (
                    df[
                        "direction"
                    ]
                    == side
                )
            ]

            stats = strategy_stats(
                subset[
                    "net_return"
                ]
            )

            rows.append(
                {
                    "regime_type":
                        column,

                    "regime":
                        regime,

                    "side":
                        side,

                    **stats
                }
            )

    return pd.DataFrame(
        rows
    )


regime_results = pd.concat(
    [
        regime_table(
            trade_regime,
            "vol_regime"
        ),

        regime_table(
            trade_regime,
            "trend_regime"
        ),

        regime_table(
            trade_regime,
            "trend_direction"
        ),

        regime_table(
            trade_regime,
            "session"
        ),
    ],
    ignore_index=True
)


regime_show = (
    regime_results.copy()
)

regime_show[
    "win_rate"
] *= 100

regime_show[
    "avg_return"
] *= 100


print()
print(
    "=================================="
)

print(
    "Regime別 BUY / SELL"
)

print(
    "=================================="
)


print(
    regime_show[
        [
            "regime_type",
            "regime",
            "side",
            "trades",
            "win_rate",
            "avg_return",
            "profit_factor"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 11. Bootstrap
#
# SELLの21件程度の成績が
# どれくらい不安定なのか確認
# ============================================================

def bootstrap_returns(
    returns,
    n_boot=10000,
    seed=42
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) < 2:

        return {
            "n":
                len(r),

            "mean":
                np.nan,

            "ci_2_5":
                np.nan,

            "ci_50":
                np.nan,

            "ci_97_5":
                np.nan,

            "prob_mean_positive":
                np.nan
        }

    rng = np.random.default_rng(
        seed
    )

    means = np.empty(
        n_boot
    )

    for i in range(
        n_boot
    ):

        sample = rng.choice(
            r,
            size=len(r),
            replace=True
        )

        means[
            i
        ] = (
            sample.mean()
        )

    return {
        "n":
            len(r),

        "mean":
            r.mean(),

        "ci_2_5":
            np.quantile(
                means,
                0.025
            ),

        "ci_50":
            np.quantile(
                means,
                0.50
            ),

        "ci_97_5":
            np.quantile(
                means,
                0.975
            ),

        "prob_mean_positive":
            (
                means
                > 0
            ).mean()
    }


bootstrap_rows = []

for side in [
    "BUY",
    "SELL"
]:

    r = trades.loc[
        trades[
            "direction"
        ]
        == side,
        "net_return"
    ]

    result = bootstrap_returns(
        r
    )

    bootstrap_rows.append(
        {
            "side":
                side,

            **result
        }
    )


bootstrap_df = pd.DataFrame(
    bootstrap_rows
)


bootstrap_show = bootstrap_df.copy()

for col in [
    "mean",
    "ci_2_5",
    "ci_50",
    "ci_97_5"
]:

    bootstrap_show[
        col
    ] *= 100


bootstrap_show[
    "prob_mean_positive"
] *= 100


print()
print(
    "=================================="
)

print(
    "Bootstrap 平均リターン95%区間"
)

print(
    "=================================="
)


print(
    bootstrap_show.to_string(
        index=False
    )
)


# ============================================================
# 12. BUY / SELLのFold安定性
# ============================================================

print()
print(
    "=================================="
)

print(
    "Fold安定性"
)

print(
    "=================================="
)


for side in [
    "buy",
    "sell"
]:

    valid = fold_results.loc[
        fold_results[
            f"{side}_trades"
        ]
        > 0
    ]

    positive_folds = (
        valid[
            f"{side}_avg_return"
        ]
        > 0
    ).sum()

    pf_above_one = (
        valid[
            f"{side}_profit_factor"
        ]
        > 1
    ).sum()

    print()

    print(
        side.upper()
    )

    print(
        "評価Fold数:",
        len(valid)
    )

    print(
        "平均リターンプラスFold:",
        positive_folds,
        "/",
        len(valid)
    )

    print(
        "PF > 1 Fold:",
        pf_above_one,
        "/",
        len(valid)
    )


# ============================================================
# 13. グラフ1
# Fold別 BUY / SELL平均リターン
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

x = np.arange(
    len(
        fold_results
    )
)

width = 0.35

plt.bar(
    x - width / 2,
    fold_results[
        "buy_avg_return"
    ]
    * 100,
    width,
    label="BUY"
)

plt.bar(
    x + width / 2,
    fold_results[
        "sell_avg_return"
    ]
    * 100,
    width,
    label="SELL"
)

plt.axhline(
    0,
    linewidth=1
)

plt.xticks(
    x,
    fold_results[
        "fold"
    ]
)

plt.xlabel(
    "Fold"
)

plt.ylabel(
    "Average net return / trade (%)"
)

plt.title(
    "BUY vs SELL: Fold-level return"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 14. グラフ2
# p_up - p_down分布
# ============================================================

direction_edge = (
    predictions[
        "p_up"
    ]
    -
    predictions[
        "p_down"
    ]
)

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.hist(
    direction_edge,
    bins=40
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel(
    "p_up - p_down"
)

plt.ylabel(
    "Count"
)

plt.title(
    "Direction model probability bias"
)

plt.tight_layout()

plt.show()


# ============================================================
# 15. CSVとして保存
# ============================================================

output_dir = (
    Path.cwd()
    /
    "buy_sell_diagnostics"
)

output_dir.mkdir(
    exist_ok=True
)


fold_results.to_csv(
    output_dir
    / "fold_buy_sell.csv",
    index=False
)

overall.to_csv(
    output_dir
    / "overall_buy_sell.csv",
    index=False
)

predictions.to_csv(
    output_dir
    / "all_test_predictions.csv",
    index=False
)

trade_regime.to_csv(
    output_dir
    / "trades_with_regime.csv",
    index=False
)

regime_results.to_csv(
    output_dir
    / "regime_buy_sell.csv",
    index=False
)

bootstrap_df.to_csv(
    output_dir
    / "bootstrap_buy_sell.csv",
    index=False
)


print()
print(
    "=================================="
)

print(
    "今回の診断完了"
)

print(
    "=================================="
)

print(
    "保存先:"
)

print(
    output_dir.resolve()
)

print()
print(
    "特に次の数字を確認:"
)

print(
    "1. SELLが何Foldでプラスか"
)

print(
    "2. SELL Bootstrap 95%区間"
)

print(
    "3. BUY/SELLシグナル数の偏り"
)

print(
    "4. SELLが強いRegime"
)

print(
    "5. 実際の市場上昇率と平均p_up"
)

## 元のセル index 21


In [ ]:
# ============================================================
# SELL閾値スイープ + BUY/SELL別閾値 Walk-Forward検証
#
# 目的
# 1. SELL閾値を下げるとSELL数がどれだけ増えるか
# 2. SELL数を増やしても平均リターン・PFが維持されるか
# 3. BUYとSELLの閾値を別々に最適化した方が良いか
#
# 前提:
# 以前のセルで以下が定義済み
#
# load_bars
# prepare_data
# outer_folds
# fit_base_models
# predict_base_models
# run_backtest
# strategy_stats
# HORIZON_BARS
# MOVE_PROB_LIST
# TP_LIST
# SL_LIST
# MIN_VALIDATION_TRADES
# ============================================================


from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 1. 実験設定
# ============================================================

# BUY側は今まで通り高め中心
BUY_DIRECTION_THRESHOLDS = [
    0.55,
    0.60,
    0.65,
    0.70,
]

# SELL側は意図的に低めまで広げる
SELL_DIRECTION_THRESHOLDS = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

# MOVE閾値
MOVE_THRESHOLDS = [
    0.55,
    0.60,
    0.65,
    0.70,
]

# TP / SL
TP_VALUES = [
    0.0005,
    0.0008,
    0.0010,
]

SL_VALUES = [
    0.0005,
    0.0007,
    0.0010,
]

# Validationで最低何件のSELLが必要か
MIN_SELL_VALIDATION_TRADES = 8

# BUY側
MIN_BUY_VALIDATION_TRADES = 15


# ============================================================
# 2. 最新CSVを探す
# ============================================================

def find_latest_snapshot():

    root = (
        Path.cwd()
        / "fx_experiment_runs"
    )

    files = list(
        root.glob(
            "*/usdjpy_5m.csv"
        )
    )

    if not files:
        raise FileNotFoundError(
            "fx_experiment_runs内にusdjpy_5m.csvがありません"
        )

    return max(
        files,
        key=lambda p:
            p.stat().st_mtime
    )


csv_path = find_latest_snapshot()

print(
    "使用CSV:"
)

print(
    csv_path
)


# ============================================================
# 3. データ読み込み
# ============================================================

bars = load_bars(
    csv_path
)

data = prepare_data(
    bars
)

print()

print(
    "5分足:",
    len(bars)
)

print(
    "使用可能データ:",
    len(data)
)


# ============================================================
# 4. BUY/SELL別シグナル関数
# ============================================================

def make_asymmetric_signals(
    p_move,
    p_up,
    p_down,
    move_threshold,
    buy_threshold,
    sell_threshold,
):
    """
    BUYとSELLで別のDirection閾値を使う。
    """

    buy = (
        (p_move >= move_threshold)
        &
        (p_up >= buy_threshold)
        &
        (p_up > p_down)
    )

    sell = (
        (p_move >= move_threshold)
        &
        (p_down >= sell_threshold)
        &
        (p_down > p_up)
    )

    signals = np.zeros(
        len(p_move)
    )

    signals[
        buy
    ] = 1

    signals[
        sell
    ] = -1

    return signals


# ============================================================
# 5. ValidationでSELL専用設定を選ぶ
# ============================================================

def choose_sell_setting(
    bars,
    validation,
    probabilities,
):

    p_move, p_up, p_down = probabilities

    val_end = (
        validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    best = None
    best_score = -np.inf

    for (
        move_threshold,
        sell_threshold,
        tp,
        sl,
    ) in itertools.product(
        MOVE_THRESHOLDS,
        SELL_DIRECTION_THRESHOLDS,
        TP_VALUES,
        SL_VALUES,
    ):

        # SELLだけ発生させる
        signals = np.zeros(
            len(validation)
        )

        sell_mask = (
            (p_move >= move_threshold)
            &
            (p_down >= sell_threshold)
            &
            (p_down > p_up)
        )

        signals[
            sell_mask
        ] = -1

        trades = run_backtest(
            bars,
            validation,
            signals,
            tp,
            sl,
            end_time=
                val_end
        )

        if (
            len(trades)
            <
            MIN_SELL_VALIDATION_TRADES
        ):
            continue

        returns = (
            trades[
                "net_return"
            ]
            .to_numpy()
        )

        avg_return = (
            returns.mean()
        )

        # 件数も評価に入れる
        score = (
            avg_return
            *
            np.sqrt(
                len(returns)
            )
        )

        if score > best_score:

            best_score = score

            best = {
                "move_threshold":
                    move_threshold,

                "sell_threshold":
                    sell_threshold,

                "tp":
                    tp,

                "sl":
                    sl,

                "validation_trades":
                    len(trades),

                "validation_avg_return":
                    avg_return,

                "validation_pf":
                    strategy_stats(
                        returns
                    )[
                        "profit_factor"
                    ],

                "score":
                    score,
            }

    return best


# ============================================================
# 6. ValidationでBUY専用設定を選ぶ
# ============================================================

def choose_buy_setting(
    bars,
    validation,
    probabilities,
):

    p_move, p_up, p_down = probabilities

    val_end = (
        validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    best = None
    best_score = -np.inf

    for (
        move_threshold,
        buy_threshold,
        tp,
        sl,
    ) in itertools.product(
        MOVE_THRESHOLDS,
        BUY_DIRECTION_THRESHOLDS,
        TP_VALUES,
        SL_VALUES,
    ):

        signals = np.zeros(
            len(validation)
        )

        buy_mask = (
            (p_move >= move_threshold)
            &
            (p_up >= buy_threshold)
            &
            (p_up > p_down)
        )

        signals[
            buy_mask
        ] = 1

        trades = run_backtest(
            bars,
            validation,
            signals,
            tp,
            sl,
            end_time=
                val_end
        )

        if (
            len(trades)
            <
            MIN_BUY_VALIDATION_TRADES
        ):
            continue

        returns = (
            trades[
                "net_return"
            ]
            .to_numpy()
        )

        avg_return = (
            returns.mean()
        )

        score = (
            avg_return
            *
            np.sqrt(
                len(returns)
            )
        )

        if score > best_score:

            best_score = score

            best = {
                "move_threshold":
                    move_threshold,

                "buy_threshold":
                    buy_threshold,

                "tp":
                    tp,

                "sl":
                    sl,

                "validation_trades":
                    len(trades),

                "validation_avg_return":
                    avg_return,

                "validation_pf":
                    strategy_stats(
                        returns
                    )[
                        "profit_factor"
                    ],

                "score":
                    score,
            }

    return best


# ============================================================
# 7. SELL閾値スイープ診断
#
# ここではValidation/Testを分けず
# "どの閾値で何件になるか"を見るための
# 診断表をFoldごとに作る
# ============================================================

threshold_diagnostic_rows = []


# ============================================================
# 8. Walk-Forward
# ============================================================

fold_rows = []

all_buy_trades = []

all_sell_trades = []


for fold in outer_folds(
    data
):

    print()

    print(
        "===================================="
    )

    print(
        f"Fold {fold.number}"
    )

    print(
        "===================================="
    )

    if (
        len(fold.train) < 500
        or
        len(fold.validation) == 0
        or
        len(fold.test) == 0
    ):

        print(
            "データ不足でskip"
        )

        continue

    # --------------------------------------------------------
    # Validation用Baseモデル
    # --------------------------------------------------------

    validation_model = (
        fit_base_models(
            fold.core,
            trees=300
        )
    )

    if validation_model is None:

        print(
            "Validationモデル学習不可"
        )

        continue

    validation_prob = (
        predict_base_models(
            validation_model,
            fold.validation
        )
    )

    # --------------------------------------------------------
    # BUY / SELLを別々に最適化
    # --------------------------------------------------------

    buy_setting = (
        choose_buy_setting(
            bars,
            fold.validation,
            validation_prob
        )
    )

    sell_setting = (
        choose_sell_setting(
            bars,
            fold.validation,
            validation_prob
        )
    )

    print()

    print(
        "BUY設定:"
    )

    print(
        buy_setting
    )

    print(
        "SELL設定:"
    )

    print(
        sell_setting
    )

    # --------------------------------------------------------
    # SELL閾値ごとのValidation診断
    # --------------------------------------------------------

    (
        val_p_move,
        val_p_up,
        val_p_down,
    ) = validation_prob

    val_end = (
        fold.validation.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    for sell_threshold in (
        SELL_DIRECTION_THRESHOLDS
    ):

        # ここでは比較しやすいように
        # MOVE=0.55
        # TP=0.0008
        # SL=0.0007
        # に一旦固定

        signals = np.zeros(
            len(
                fold.validation
            )
        )

        mask = (
            (val_p_move >= 0.55)
            &
            (val_p_down >= sell_threshold)
            &
            (val_p_down > val_p_up)
        )

        signals[
            mask
        ] = -1

        trades = run_backtest(
            bars,
            fold.validation,
            signals,
            0.0008,
            0.0007,
            end_time=
                val_end
        )

        stats = (
            strategy_stats(
                trades[
                    "net_return"
                ]
            )
        )

        threshold_diagnostic_rows.append(
            {
                "fold":
                    fold.number,

                "sell_threshold":
                    sell_threshold,

                **stats
            }
        )

    # --------------------------------------------------------
    # Test用最終Baseモデル
    # --------------------------------------------------------

    final_model = (
        fit_base_models(
            fold.train,
            trees=400
        )
    )

    if final_model is None:

        continue

    (
        test_p_move,
        test_p_up,
        test_p_down,
    ) = predict_base_models(
        final_model,
        fold.test
    )

    test_end = (
        fold.test.index[-1]
        +
        pd.Timedelta(
            minutes=5
        )
    )

    row = {
        "fold":
            fold.number
    }

    # ========================================================
    # BUY Test
    # ========================================================

    if buy_setting is not None:

        buy_signals = np.zeros(
            len(
                fold.test
            )
        )

        buy_mask = (
            (
                test_p_move
                >=
                buy_setting[
                    "move_threshold"
                ]
            )
            &
            (
                test_p_up
                >=
                buy_setting[
                    "buy_threshold"
                ]
            )
            &
            (
                test_p_up
                >
                test_p_down
            )
        )

        buy_signals[
            buy_mask
        ] = 1

        buy_trades = run_backtest(
            bars,
            fold.test,
            buy_signals,
            buy_setting[
                "tp"
            ],
            buy_setting[
                "sl"
            ],
            end_time=
                test_end
        )

        buy_trades[
            "fold"
        ] = (
            fold.number
        )

        all_buy_trades.append(
            buy_trades
        )

        buy_stats = (
            strategy_stats(
                buy_trades[
                    "net_return"
                ]
            )
        )

        for key, value in (
            buy_stats.items()
        ):

            row[
                f"buy_{key}"
            ] = value

        row[
            "buy_move_threshold"
        ] = (
            buy_setting[
                "move_threshold"
            ]
        )

        row[
            "buy_direction_threshold"
        ] = (
            buy_setting[
                "buy_threshold"
            ]
        )

    # ========================================================
    # SELL Test
    # ========================================================

    if sell_setting is not None:

        sell_signals = np.zeros(
            len(
                fold.test
            )
        )

        sell_mask = (
            (
                test_p_move
                >=
                sell_setting[
                    "move_threshold"
                ]
            )
            &
            (
                test_p_down
                >=
                sell_setting[
                    "sell_threshold"
                ]
            )
            &
            (
                test_p_down
                >
                test_p_up
            )
        )

        sell_signals[
            sell_mask
        ] = -1

        sell_trades = run_backtest(
            bars,
            fold.test,
            sell_signals,
            sell_setting[
                "tp"
            ],
            sell_setting[
                "sl"
            ],
            end_time=
                test_end
        )

        sell_trades[
            "fold"
        ] = (
            fold.number
        )

        all_sell_trades.append(
            sell_trades
        )

        sell_stats = (
            strategy_stats(
                sell_trades[
                    "net_return"
                ]
            )
        )

        for key, value in (
            sell_stats.items()
        ):

            row[
                f"sell_{key}"
            ] = value

        row[
            "sell_move_threshold"
        ] = (
            sell_setting[
                "move_threshold"
            ]
        )

        row[
            "sell_direction_threshold"
        ] = (
            sell_setting[
                "sell_threshold"
            ]
        )

    fold_rows.append(
        row
    )


# ============================================================
# 9. 結合
# ============================================================

fold_results = (
    pd.DataFrame(
        fold_rows
    )
)

threshold_diagnostics = (
    pd.DataFrame(
        threshold_diagnostic_rows
    )
)


if all_buy_trades:

    buy_trades_all = (
        pd.concat(
            all_buy_trades,
            ignore_index=True
        )
    )

else:

    buy_trades_all = (
        pd.DataFrame()
    )


if all_sell_trades:

    sell_trades_all = (
        pd.concat(
            all_sell_trades,
            ignore_index=True
        )
    )

else:

    sell_trades_all = (
        pd.DataFrame()
    )


# ============================================================
# 10. BUY / SELL最終比較
# ============================================================

print()

print(
    "===================================="
)

print(
    "BUY / SELL 別閾値 最終結果"
)

print(
    "===================================="
)


comparison_rows = []


for (
    side,
    trades_df,
) in [
    (
        "BUY",
        buy_trades_all
    ),
    (
        "SELL",
        sell_trades_all
    ),
]:

    if trades_df.empty:

        stats = (
            strategy_stats(
                []
            )
        )

    else:

        stats = (
            strategy_stats(
                trades_df[
                    "net_return"
                ]
            )
        )

    comparison_rows.append(
        {
            "side":
                side,

            **stats
        }
    )


comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


shown = (
    comparison.copy()
)


for col in [
    "win_rate",
    "avg_return",
    "max_dd",
    "total_growth",
]:

    shown[
        col
    ] *= 100


print(
    shown.to_string(
        index=False
    )
)


# ============================================================
# 11. SELL閾値別の平均結果
# ============================================================

print()

print(
    "===================================="
)

print(
    "SELL閾値別 Validation平均"
)

print(
    "===================================="
)


threshold_summary = (
    threshold_diagnostics
    .groupby(
        "sell_threshold"
    )
    .agg(
        folds=(
            "fold",
            "nunique"
        ),

        avg_trades=(
            "trades",
            "mean"
        ),

        total_trades=(
            "trades",
            "sum"
        ),

        avg_win_rate=(
            "win_rate",
            "mean"
        ),

        avg_return=(
            "avg_return",
            "mean"
        ),

        avg_pf=(
            "profit_factor",
            "mean"
        ),

        avg_max_dd=(
            "max_dd",
            "mean"
        ),
    )
    .reset_index()
)


threshold_show = (
    threshold_summary.copy()
)


threshold_show[
    "avg_win_rate"
] *= 100


threshold_show[
    "avg_return"
] *= 100


threshold_show[
    "avg_max_dd"
] *= 100


print(
    threshold_show.to_string(
        index=False
    )
)


# ============================================================
# 12. Fold別 SELL結果
# ============================================================

print()

print(
    "===================================="
)

print(
    "Fold別SELL"
)

print(
    "===================================="
)


sell_columns = [
    "fold",
    "sell_move_threshold",
    "sell_direction_threshold",
    "sell_trades",
    "sell_win_rate",
    "sell_avg_return",
    "sell_profit_factor",
    "sell_max_dd",
]


sell_fold_table = (
    fold_results[
        [
            c
            for c
            in sell_columns
            if c
            in fold_results.columns
        ]
    ]
    .copy()
)


for col in [
    "sell_win_rate",
    "sell_avg_return",
    "sell_max_dd",
]:

    if (
        col
        in sell_fold_table.columns
    ):

        sell_fold_table[
            col
        ] *= 100


print(
    sell_fold_table.to_string(
        index=False
    )
)


# ============================================================
# 13. SELLサンプル数が本当に増えたか
# ============================================================

print()

print(
    "===================================="
)

print(
    "SELL件数評価"
)

print(
    "===================================="
)


sell_count = (
    len(
        sell_trades_all
    )
)


print(
    "Test SELL総取引数:",
    sell_count
)


if sell_count < 50:

    print(
        "まだ少ないです。"
    )

    print(
        "現状ではSELL edgeの信頼性は低いです。"
    )

elif sell_count < 100:

    print(
        "サンプルは増えましたが、まだ予備検証段階です。"
    )

elif sell_count < 300:

    print(
        "SELLの統計検証を進められる水準に近づいています。"
    )

else:

    print(
        "SELLサンプル数はかなり改善しています。"
    )


# ============================================================
# 14. Fold安定性
# ============================================================

if (
    not sell_trades_all.empty
):

    positive_folds = 0

    pf_above_one = 0

    evaluated_folds = 0

    for _, row in (
        fold_results.iterrows()
    ):

        if (
            "sell_trades"
            not in row
            or
            pd.isna(
                row[
                    "sell_trades"
                ]
            )
            or
            row[
                "sell_trades"
            ]
            <= 0
        ):

            continue

        evaluated_folds += 1

        if (
            row[
                "sell_avg_return"
            ]
            > 0
        ):

            positive_folds += 1

        if (
            row[
                "sell_profit_factor"
            ]
            > 1
        ):

            pf_above_one += 1

    print()

    print(
        "SELL評価Fold:",
        evaluated_folds
    )

    print(
        "平均リターンプラスFold:",
        positive_folds,
        "/",
        evaluated_folds
    )

    print(
        "PF > 1 Fold:",
        pf_above_one,
        "/",
        evaluated_folds
    )


# ============================================================
# 15. Bootstrap
# ============================================================

def bootstrap_mean(
    returns,
    n_boot=10000,
    seed=42,
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) < 2:

        return {
            "n":
                len(r),

            "mean":
                np.nan,

            "ci_2_5":
                np.nan,

            "ci_50":
                np.nan,

            "ci_97_5":
                np.nan,

            "prob_positive":
                np.nan,
        }

    rng = (
        np.random.default_rng(
            seed
        )
    )

    boot_means = (
        np.empty(
            n_boot
        )
    )

    for i in range(
        n_boot
    ):

        sample = rng.choice(
            r,
            size=len(r),
            replace=True
        )

        boot_means[
            i
        ] = (
            sample.mean()
        )

    return {
        "n":
            len(r),

        "mean":
            r.mean(),

        "ci_2_5":
            np.quantile(
                boot_means,
                0.025
            ),

        "ci_50":
            np.quantile(
                boot_means,
                0.50
            ),

        "ci_97_5":
            np.quantile(
                boot_means,
                0.975
            ),

        "prob_positive":
            (
                boot_means
                > 0
            ).mean(),
    }


if (
    not sell_trades_all.empty
):

    sell_bootstrap = (
        bootstrap_mean(
            sell_trades_all[
                "net_return"
            ]
        )
    )

    print()

    print(
        "===================================="
    )

    print(
        "SELL Bootstrap"
    )

    print(
        "===================================="
    )

    print(
        "N:",
        sell_bootstrap[
            "n"
        ]
    )

    print(
        "平均リターン:",
        sell_bootstrap[
            "mean"
        ]
        * 100,
        "%"
    )

    print(
        "95%CI:",
        sell_bootstrap[
            "ci_2_5"
        ]
        * 100,
        "%",
        "～",
        sell_bootstrap[
            "ci_97_5"
        ]
        * 100,
        "%"
    )

    print(
        "平均>0確率:",
        sell_bootstrap[
            "prob_positive"
        ]
        * 100,
        "%"
    )


# ============================================================
# 16. グラフ
# SELL閾値と取引数
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    threshold_summary[
        "sell_threshold"
    ],
    threshold_summary[
        "total_trades"
    ],
    marker="o"
)

plt.xlabel(
    "SELL direction threshold"
)

plt.ylabel(
    "Validation SELL trades"
)

plt.title(
    "SELL threshold vs trade count"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ============================================================
# 17. グラフ
# SELL閾値と平均リターン
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    threshold_summary[
        "sell_threshold"
    ],
    threshold_summary[
        "avg_return"
    ]
    * 100,
    marker="o"
)

plt.axhline(
    0,
    linewidth=1
)

plt.xlabel(
    "SELL direction threshold"
)

plt.ylabel(
    "Average net return (%)"
)

plt.title(
    "SELL threshold vs average return"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ============================================================
# 18. 保存
# ============================================================

output_dir = (
    Path.cwd()
    /
    "sell_threshold_experiment"
)

output_dir.mkdir(
    exist_ok=True
)


fold_results.to_csv(
    output_dir
    / "fold_results.csv",
    index=False
)

threshold_diagnostics.to_csv(
    output_dir
    / "threshold_diagnostics.csv",
    index=False
)

threshold_summary.to_csv(
    output_dir
    / "threshold_summary.csv",
    index=False
)

comparison.to_csv(
    output_dir
    / "buy_sell_comparison.csv",
    index=False
)

if (
    not buy_trades_all.empty
):

    buy_trades_all.to_csv(
        output_dir
        / "buy_trades.csv",
        index=False
    )

if (
    not sell_trades_all.empty
):

    sell_trades_all.to_csv(
        output_dir
        / "sell_trades.csv",
        index=False
    )


print()

print(
    "===================================="
)

print(
    "実験完了"
)

print(
    "===================================="
)

print(
    "保存先:"
)

print(
    output_dir.resolve()
)

print()

print(
    "次に確認する項目"
)

print(
    "1. SELL総取引数が21件からどれだけ増えたか"
)

print(
    "2. SELL閾値を下げても平均リターンがプラスか"
)

print(
    "3. SELL PFが1より上を維持するか"
)

print(
    "4. 複数Foldでプラスか"
)

print(
    "5. Bootstrap 95%CIが0をまたぐか"
)